# Практическое задание «Оценка точности модели, переобучение, регуляризация»

В этом ноутбуке вы:
1. обучите базовую логистическую регрессию;
2. оцените её качество с помощью `Accuracy`, `Precision`, `Recall`, `F1`, `ROC-AUC`, `PR-AUC`;
3. проанализируете матрицу ошибок;
4. сравните несколько способов кросс-валидации;
5. сравните L1- и L2-регуляризацию.

Во всех частях используется один и тот же датасет **Fair / Affairs** из `statsmodels`.

## Что уже подготовлено в шаблоне

Чтобы вы могли сосредоточиться на логике задания, в файле уже есть:
- все импорты;
- загрузка датасета;
- каркас графиков;
- вспомогательные функции;
- места, где нужно дописать код;
- вопросы для коротких выводов.

Ваша задача — заполнить отмеченные места и кратко интерпретировать результаты.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)

In [ ]:
def build_model(penalty="l2", C=1.0):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            penalty=penalty,
            C=C
        ))
    ])

def plot_confusion_heatmap(cm, labels=("Нет измены", "Была измена")):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Предсказанный класс")
    ax.set_ylabel("Истинный класс")
    ax.set_title("Матрица ошибок")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.show()

## 1. Загрузка данных и минимальная подготовка

Используйте датасет **Fair / Affairs** из `statsmodels`.
Целевая переменная должна быть бинарной:

- `1`, если `affairs > 0`
- `0`, если `affairs == 0`

Обратите внимание: в исходном датасете признаки уже числовые, поэтому отдельное кодирование категориальных признаков здесь не требуется.

In [ ]:
df = sm.datasets.fair.load_pandas().data.copy()
df.head()

In [ ]:
# TODO:
# 1. создайте бинарную целевую переменную affair_flag
# 2. посчитайте количество пропусков
# 3. посчитайте долю положительного класса

df["affair_flag"] = ...
missing_values = ...
positive_share = ...

print("Размер датасета:", df.shape)
print("Суммарное число пропусков:", missing_values)
print("Доля положительного класса:", positive_share)
display(df.head())

### Короткий вывод

Ответьте кратко:
1. Есть ли в данных пропуски?
2. Насколько сбалансирован целевой класс?
3. Почему для этого урока достаточно минимальной подготовки данных?

In [ ]:
# TODO:
# 1. сформируйте матрицу признаков X
# 2. сформируйте целевую переменную y
# 3. уберите из X исходный столбец affairs и созданную цель affair_flag

X = ...
y = ...

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 2. Базовая логистическая регрессия и основные метрики

Обучите базовую модель логистической регрессии и оцените её качество.
Посчитайте:

- `accuracy`
- `precision`
- `recall`
- `f1-score`

Также выведите метрики для **обоих классов**.

In [ ]:
# TODO:
# 1. создайте базовую модель
# 2. обучите её на train
# 3. получите предсказания классов и вероятности положительного класса

baseline_model = ...
baseline_model.fit(...)

y_pred = ...
y_proba = ...

In [ ]:
# TODO:
# Посчитайте основные метрики качества

accuracy = ...
precision = ...
recall = ...
f1 = ...

metrics_table = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall", "f1"],
    "value": [accuracy, precision, recall, f1]
})

display(metrics_table)

# TODO:
# Получите таблицу с метриками для обоих классов
report = ...
display(report)

### Короткий вывод

Ответьте кратко:
1. Какая метрика получилась выше всего?
2. Какая метрика сильнее всего проседает?
3. Что это говорит о поведении модели на положительном классе?

## 3. Матрица ошибок

Постройте матрицу ошибок и проанализируйте:
- сколько у модели ложноположительных ошибок;
- сколько ложноотрицательных ошибок;
- какой тип ошибок встречается чаще.

In [ ]:
# TODO:
cm = ...
print(cm)
plot_confusion_heatmap(cm)

### Короткий вывод

Ответьте кратко:
1. Какие ошибки встречаются чаще — ложноположительные или ложноотрицательные?
2. Почему одной accuracy недостаточно, чтобы это увидеть?

## 4. ROC-кривая и Precision-Recall кривая

Используя вероятности положительного класса:

1. постройте ROC-кривую и посчитайте `ROC-AUC`;
2. постройте Precision-Recall curve и посчитайте `Average Precision (AP)`.

Затем кратко сравните, что показывают эти две метрики.


In [ ]:
# TODO:
# ROC-кривая и ROC-AUC
fpr, tpr, roc_thresholds = ...
roc_auc = ...

# TODO:
# PR-кривая и Average Precision (AP)
precision_curve, recall_curve, pr_thresholds = ...
ap_score = ...

print("ROC-AUC:", roc_auc)
print("Average Precision (AP):", ap_score)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--")
axes[0].set_title("ROC curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(recall_curve, precision_curve, label=f"PR-AUC = {pr_auc:.3f}")
axes[1].set_title("Precision-Recall curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
axes[1].grid(True)

plt.show()

### Короткий вывод

Ответьте кратко:
1. Что показывает ROC-AUC?
2. Что показывает Average Precision (AP)?
3. Почему AP часто полезнее, когда нас особенно интересует положительный класс?


## 5. Кросс-валидация

Сравните устойчивость качества модели тремя способами:

1. несколько обычных train/test split с разными `random_state`;
2. `KFold`;
3. `StratifiedKFold`.

Для сравнения используйте метрику **F1**.

In [ ]:
# TODO:
# Проведите 5 обычных разбиений train/test с random_state от 0 до 4
# Для каждого разбиения:
# 1. обучите новую модель
# 2. посчитайте F1 на test
# 3. сохраните результат

holdout_scores = []

for rs in range(5):
    X_train_tmp, X_test_tmp, y_train_tmp, y_test_tmp = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=rs,
        stratify=y
    )

    model_tmp = ...
    model_tmp.fit(...)
    pred_tmp = ...
    score_tmp = ...
    holdout_scores.append(score_tmp)

holdout_scores = np.array(holdout_scores)
holdout_scores

In [ ]:
# TODO:
# Посчитайте F1 через KFold и StratifiedKFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kfold_scores = ...
skfold_scores = ...

cv_summary = pd.DataFrame({
    "method": ["Repeated holdout", "KFold", "StratifiedKFold"],
    "mean_f1": [holdout_scores.mean(), kfold_scores.mean(), skfold_scores.mean()],
    "std_f1": [holdout_scores.std(), kfold_scores.std(), skfold_scores.std()]
})

display(cv_summary)

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot([holdout_scores, kfold_scores, skfold_scores], labels=["Holdout", "KFold", "StratifiedKFold"])
plt.title("Сравнение F1 при разных способах валидации")
plt.ylabel("F1")
plt.grid(True)
plt.show()

### Короткий вывод

Ответьте кратко:
1. У какого способа оценки метрика самая стабильная?
2. Почему `StratifiedKFold` часто предпочтительнее для задач классификации?

## 6. Сравнение L1- и L2-регуляризации

Чтобы не перегружать задание, сравните только 4 конфигурации:

- L1, `C = 0.01`
- L1, `C = 1`
- L2, `C = 0.01`
- L2, `C = 1`

Для каждой модели посчитайте:
- `train_accuracy`
- `test_accuracy`
- `train_f1`
- `test_f1`
- `train_roc_auc`
- `test_roc_auc`

Сравните результаты и сделайте вывод, есть ли признаки переобучения или недообучения.

In [ ]:
configs = [
    ("l1", 0.01),
    ("l1", 1.0),
    ("l2", 0.01),
    ("l2", 1.0),
]
configs

In [ ]:
# TODO:
# Пройдитесь циклом по configs, обучите модель для каждой конфигурации
# и сохраните метрики в список словарей

reg_results = []

for penalty, C in configs:
    model = ...
    model.fit(...)

    train_pred = ...
    test_pred = ...

    train_proba = ...
    test_proba = ...

    reg_results.append({
        "penalty": penalty,
        "C": C,
        "train_accuracy": ...,
        "test_accuracy": ...,
        "train_f1": ...,
        "test_f1": ...,
        "train_roc_auc": ...,
        "test_roc_auc": ...,
    })

reg_results = pd.DataFrame(reg_results)
display(reg_results)

In [ ]:
plt.figure(figsize=(8, 5))
labels = reg_results["penalty"] + ", C=" + reg_results["C"].astype(str)
x = np.arange(len(labels))
width = 0.35

plt.bar(x - width/2, reg_results["train_f1"], width=width, label="Train F1")
plt.bar(x + width/2, reg_results["test_f1"], width=width, label="Test F1")

plt.xticks(x, labels, rotation=20)
plt.ylabel("F1")
plt.title("Сравнение F1 для разных вариантов регуляризации")
plt.legend()
plt.grid(True, axis="y")
plt.show()

### Короткий вывод

Ответьте кратко:
1. Какая конфигурация даёт лучший результат на test?
2. Где сильнее заметно недообучение?
3. Есть ли явные признаки переобучения?
4. Какая регуляризация в этой задаче выглядит более удачной и почему?

# Итоговый вывод

Напишите 5–8 предложений, в которых кратко ответите:

1. Почему accuracy недостаточно как единственная метрика?
2. Что дополнительно дали F1, ROC-AUC и Average Precision (AP)?
3. Зачем нужна кросс-валидация?
4. Как регуляризация помогает контролировать переобучение?
5. Какая модель в этой работе показала себя лучше всего?
